# The Middleware Lifecycle

Every middleware hook runs at a specific point in the agent loop. Understanding the order is the key to combining them correctly.

```mermaid
flowchart TD
    A[before_agent] --> B[before_model]
    B --> C[wrap_model_call -> MODEL -> wrap_model_call]
    C --> D[after_model]
    D -->|tool calls?| E[wrap_tool_call -> TOOL]
    E --> B
    D -->|done| F[after_agent]
```

- `before_agent` / `after_agent` run **once** per run (start and end).
- `before_model` / `wrap_model_call` / `after_model` run on **every** model turn.
- `wrap_tool_call` wraps **each** tool execution.

The cells below attach a logger to each hook so you can watch the real order.

In [1]:
%run ../../langchain_common.py

C:\Users\akhawaja\git\cs4603\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.


## Logging middleware
Each hook just prints when it fires. `wrap_*` hooks print on both sides of the call they wrap.

In [3]:
from langchain.agents.middleware import (
    before_agent, before_model, after_model, after_agent,
    wrap_model_call, wrap_tool_call,
)


@before_agent
def m_before_agent(state, runtime):
    print("[1] before_agent      (run starts)")
    return None


@before_model
def m_before_model(state, runtime):
    print("[2]   before_model    (each model turn)")
    return None


@wrap_model_call
def m_wrap_model(request, handler):
    print("[3]     wrap_model_call -> calling model")
    response = handler(request)
    print("[4]     wrap_model_call <- model returned")
    return response


@after_model
def m_after_model(state, runtime):
    print("[5]   after_model")
    return None


@wrap_tool_call
def m_wrap_tool(request, handler):
    print(f"[6]     wrap_tool_call -> {request.tool_call['name']}")
    message = handler(request)
    print("[7]     wrap_tool_call <- tool done")
    return message


@after_agent
def m_after_agent(state, runtime):
    print("[8] after_agent       (run ends)")
    return None

## Watch the order
The agent has one tool, so a full run does: model turn (decides to call the tool) -> tool -> model turn (final answer). Notice how `before_model`/`after_model` repeat while `before_agent`/`after_agent` fire only once.

In [4]:
@tool
def get_time(city: str) -> str:
    """Get the current time in a city."""
    return f"It is 10:30 AM in {city}."


agent = create_agent(
    model=llm_noreason,
    tools=[get_time],
    middleware=[
        m_before_agent,
        m_before_model,
        m_wrap_model,
        m_after_model,
        m_wrap_tool,
        m_after_agent,
    ],
)

resp = agent.invoke({"messages": [HumanMessage(content="What time is it in Tokyo?")]})

print("\n--- final answer ---")
resp["messages"][-1].pretty_print()

[1] before_agent      (run starts)
[2]   before_model    (each model turn)
[3]     wrap_model_call -> calling model
[4]     wrap_model_call <- model returned
[5]   after_model
[6]     wrap_tool_call -> get_time
[7]     wrap_tool_call <- tool done
[2]   before_model    (each model turn)
[3]     wrap_model_call -> calling model
[4]     wrap_model_call <- model returned
[5]   after_model
[8] after_agent       (run ends)

--- final answer ---
================================== Ai Message ==================================

It is 10:30 AM in Tokyo.


## Stacking order

When you pass a **list** of middleware, they nest like layers (an onion):

- `before_*` hooks run **top-to-bottom** (list order).
- `after_*` and the back half of `wrap_*` hooks run **bottom-to-top** (reverse order).

So the first middleware in the list is the outermost layer - it sees the request first and the response last. Order matters when one guardrail should run before another (e.g. block input before redacting it).